# Day 15 - SMOTE + KMeans + HalvingSearch


## 학습 목표
- 불균형 데이터 처리 (SMOTE, ROS)
- KMeans 기초
- HalvingGridSearch / HalvingRandomSearch


## 1. 불균형 데이터 생성 및 SMOTE 적용


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE, RandomOverSampler
import warnings
warnings.filterwarnings('ignore')

# 불균형 데이터 생성 (9:1)
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_redundant=5, weights=[0.9, 0.1], random_state=42
)
print("Original class distribution:", np.bincount(y))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

# 원본 (불균형)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
print("\n=== Without resampling ===")
print(classification_report(y_te, rf.predict(X_te), digits=4))

# SMOTE
smote = SMOTE(random_state=42)
X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)
print("SMOTE class distribution:", np.bincount(y_tr_sm))
rf_sm = RandomForestClassifier(n_estimators=100, random_state=42)
rf_sm.fit(X_tr_sm, y_tr_sm)
print("\n=== With SMOTE ===")
print(classification_report(y_te, rf_sm.predict(X_te), digits=4))


## 2. KMeans 기초


In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)

inertias, silhouettes = [], []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_iris)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_iris, labels))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_title('Elbow Method (Inertia)')
axes[0].set_xlabel('K')
axes[1].plot(K_range, silhouettes, 'ro-')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('K')
plt.tight_layout()
plt.show()
print("Best K by silhouette:", list(K_range)[np.argmax(silhouettes)])


## 3. HalvingRandomSearch (빠른 하이퍼파라미터 탐색)


In [ ]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
from scipy.stats import randint
import xgboost as xgb
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = StandardScaler().fit_transform(data.data), data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_dist = {
    'max_depth': randint(2, 8),
    'n_estimators': randint(50, 200),
    'learning_rate': [0.01, 0.05, 0.1, 0.2]
}
model = xgb.XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss')
halving = HalvingRandomSearchCV(
    model, param_dist, n_candidates=20, cv=3,
    factor=2, random_state=42, n_jobs=-1, verbose=1
)
halving.fit(X_tr, y_tr)
print("Best params:", halving.best_params_)
print("Best score :", round(halving.best_score_, 4))
print("Test Acc   :", round(accuracy_score(y_te, halving.predict(X_te)), 4))
